# Step 3 & 4: Constructing Demand Dataset & Strong Simple Baselines
## Testing the Core Philosophy: Does ML Provide Meaningful Improvement?

Before touching any Machine Learning models, we establish strong simple forecasting baselines:
1. **Baseline A: Naive Previous-Period**: $\hat{y}_t = y_{t-1}$
2. **Baseline B: Historical / Seasonal Lookup**: $\hat{y}_t = \bar{y}_{\text{zone, DOW, slot}}$ (learned strictly on Training set)
3. **Baseline C: Moving Average**: 1-hour rolling average strictly of past intervals

### Rules:
- **Strict Chronological Splitting**: Train (Jan 1–20), Val (Jan 21–25), Test (Jan 26–31).
- **NO random train_test_split**: Temporal ordering must be strictly respected to avoid future lookahead leakage.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append(str(Path.cwd().parent))
from src.data_loader import load_processed_demand, split_chronological
from src.features import create_calendar_features, create_lag_and_rolling_features
from src.baselines import NaiveLastPeriodBaseline, HistoricalSeasonalBaseline, MovingAverageBaseline
from src.metrics import calculate_metrics, build_comparison_table, evaluate_subgroups
from src.config import TRAIN_START, TRAIN_END, VAL_START, VAL_END, TEST_START, TEST_END

print("Loading 15-minute demand grid...")
grid_df = load_processed_demand()
print(f"Loaded {len(grid_df):,} rows.")


### 1. Feature Generation for Baselines (Zero-Leakage Lags)
We generate calendar features and strict backward-looking lags (`lag_1` and `rolling_mean_4`).


In [ ]:
df = create_calendar_features(grid_df)
df = create_lag_and_rolling_features(df, lag_steps=[1], rolling_windows=[4])

# Drop initial row per zone where lag_1 is NaN
df = df.dropna(subset=['lag_1', 'rolling_mean_4']).reset_index(drop=True)
print(f"Prepared dataset shape: {df.shape}")
df.head()


### 2. Chronological Train / Validation / Test Splits
- **Training**: Jan 1 to Jan 20
- **Validation**: Jan 21 to Jan 25
- **Test**: Jan 26 to Jan 31


In [ ]:
train_df, val_df, test_df = split_chronological(df)

print(f"Train set: {len(train_df):,} rows ({train_df['timestamp'].min()} to {train_df['timestamp'].max()})")
print(f"Val set:   {len(val_df):,} rows ({val_df['timestamp'].min()} to {val_df['timestamp'].max()})")
print(f"Test set:  {len(test_df):,} rows ({test_df['timestamp'].min()} to {test_df['timestamp'].max()})")


### 3. Baseline A: Naive Previous-Period Forecast
Predicts that demand in the next 15 minutes will equal demand in the previous 15 minutes:
$$\hat{y}_t = y_{t-1}$$


In [ ]:
naive_model = NaiveLastPeriodBaseline().fit(train_df)

naive_val_preds = naive_model.predict(val_df)
naive_test_preds = naive_model.predict(test_df)

val_metrics_naive = calculate_metrics(val_df['demand'].to_numpy(), naive_val_preds)
test_metrics_naive = calculate_metrics(test_df['demand'].to_numpy(), naive_test_preds)

print(f"Naive Baseline (Val):  MAE = {val_metrics_naive['MAE']:.3f}, RMSE = {val_metrics_naive['RMSE']:.3f}")
print(f"Naive Baseline (Test): MAE = {test_metrics_naive['MAE']:.3f}, RMSE = {test_metrics_naive['RMSE']:.3f}")


### 4. Baseline B: Historical / Seasonal Lookup Baseline
Computes average demand for each `(PULocationID, day_of_week, 15min_time_slot)` strictly from the training set.


In [ ]:
seasonal_model = HistoricalSeasonalBaseline().fit(train_df)

seasonal_val_preds = seasonal_model.predict(val_df)
seasonal_test_preds = seasonal_model.predict(test_df)

val_metrics_seasonal = calculate_metrics(val_df['demand'].to_numpy(), seasonal_val_preds)
test_metrics_seasonal = calculate_metrics(test_df['demand'].to_numpy(), seasonal_test_preds)

print(f"Seasonal Baseline (Val):  MAE = {val_metrics_seasonal['MAE']:.3f}, RMSE = {val_metrics_seasonal['RMSE']:.3f}")
print(f"Seasonal Baseline (Test): MAE = {test_metrics_seasonal['MAE']:.3f}, RMSE = {test_metrics_seasonal['RMSE']:.3f}")


### 5. Baseline C: Moving Average Baseline (1-Hour Rolling Window)
Computes the rolling mean of the 4 prior 15-minute intervals strictly before $t$.


In [ ]:
ma_model = MovingAverageBaseline(window=4).fit(train_df)

ma_val_preds = ma_model.predict(val_df)
ma_test_preds = ma_model.predict(test_df)

val_metrics_ma = calculate_metrics(val_df['demand'].to_numpy(), ma_val_preds)
test_metrics_ma = calculate_metrics(test_df['demand'].to_numpy(), ma_test_preds)

print(f"Moving Avg Baseline (Val):  MAE = {val_metrics_ma['MAE']:.3f}, RMSE = {val_metrics_ma['RMSE']:.3f}")
print(f"Moving Avg Baseline (Test): MAE = {test_metrics_ma['MAE']:.3f}, RMSE = {test_metrics_ma['RMSE']:.3f}")


### 6. Baseline Benchmark Comparison Table (Test Set: Jan 26–31)
Summary of all baseline performances on the unseen test period:


In [ ]:
baseline_dict = {
    "Naive (t-1)": naive_test_preds,
    "Historical Seasonal": seasonal_test_preds,
    "Moving Average (1-hour)": ma_test_preds
}

comp_table = build_comparison_table(baseline_dict, test_df['demand'].to_numpy(), baseline_key="Naive (t-1)")
print(comp_table.to_string(index=False))


### Baseline Findings & Target for Machine Learning
- The Historical Seasonal baseline captures recurrent time-of-day and day-of-week patterns, significantly outperforming the Naive model in RMSE.
- The Moving Average smooths short-term fluctuations.
- **The Challenge for Step 5 (ML Model)**: Any ML model we propose **MUST** demonstrate lower MAE and RMSE than both Naive and Historical Seasonal baselines to justify its deployment.
